1. Instalacja bibliotek

In [13]:
%pip install requests pymongo


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: /usr/local/bin/python3.14 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


2. Import bibliotek

In [14]:
import requests
from pymongo import MongoClient
from contextlib import contextmanager

3. Parametry konfiguracyjne połączenia z bazą

In [15]:
MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "crypto_db"
COLLECTION_NAME = "networks"

4. Zarządzanie połączeniem

In [16]:
@contextmanager
def mongodb_client():
    client = MongoClient(MONGO_URI)
    try:
        yield client
    finally:
        client.close()

5. Inicjalizacja kolekcji mongodb

In [17]:
def init_collection(db):
    db[COLLECTION_NAME].drop()
    print(f"Kolekcja '{COLLECTION_NAME}' została zresetowana.")

6. Zapis danych

In [18]:
def save_networks_to_mongo(db, networks_data):
    if not networks_data:
        return

    collection = db[COLLECTION_NAME]
    result = collection.insert_many(networks_data)
    print(f"Pomyślnie zapisano {len(result.inserted_ids)} sieci.")

7. Pobieranie danych z API

In [19]:
def fetch_gecko_networks():
    url = "https://api.geckoterminal.com/api/v2/networks"
    headers = {"Accept": "application/json"}

    print("Pobieranie danych z GeckoTerminal...")
    response = requests.get(url, headers=headers)
    response.raise_for_status()

    return response.json().get('data', [])

8. Agregacja danych

In [20]:
def count_networks_by_type(db):
    pipeline = [
        {
            "$group": {
                "_id": "$type",
                "count": {"$sum": 1}
            }
        },
        {
            "$sort": {"count": -1}
        }
    ]

    results = list(db[COLLECTION_NAME].aggregate(pipeline))
    return results

9. Uruchomienie aplikacji

In [21]:
def main():
    try:
        networks = fetch_gecko_networks()

        with mongodb_client() as client:
            db = client[DB_NAME]

            init_collection(db)
            save_networks_to_mongo(db, networks)

            print("\n--- WYNIKI AGREGACJI (Liczba sieci per typ) ---")
            stats = count_networks_by_type(db)

            for item in stats:
                print(f"Typ: {item['_id']} | Ilość: {item['count']}")

    except Exception as e:
        print(f"Wystąpił błąd: {e}")

if __name__ == "__main__":
    main()

Pobieranie danych z GeckoTerminal...
Kolekcja 'networks' została zresetowana.
Pomyślnie zapisano 100 sieci.

--- WYNIKI AGREGACJI (Liczba sieci per typ) ---
Typ: network | Ilość: 100
